# Chapter 35 — RPC

**Companion to *Pi Agents*** · [`Pi Agents` chapter 35](https://programmer.ie/books/pi/35-chapter/)

| | |
|---|---|
| Chapter | `35-chapter.md` · weight 35 · `/books/pi/35-chapter/` |
| Notebook | `notebooks/pi/35-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does a successful `prompt` response mean *finished*, and does the UI
subprotocol block on the client?

The chapter's point: a `prompt` response means **accepted**, not finished —
`disposition: "started"`. The run follows as events, and a client waits for
`agent_settled`, not `agent_end`. The UI subprotocol is a request/answer pair
matched by id, and it blocks the client until the answer arrives.


## What this notebook establishes

- Responses **repeat the command id**; two outstanding commands are matched by id, not by order.
- A successful `prompt` response means **accepted**: `disposition: "started"`, and it arrives **before the run's first event**.
- Session events carry **no command id** — only responses do.
- `agent_end` is not the end: **`agent_settled` follows it**, and is what a client waits for.
- A malformed JSON line produces a **parse response with no request id**; an unknown command is an error response and the process keeps serving.
- The **UI subprotocol** is a request/answer pair matched by id; the client blocks until it answers, and a UI response does **not** produce a normal command response.
- `get_entries` uses an entry id as a **durable cursor** (`since`), reports the leaf, and fails on an unknown cursor.
- `extension_error` is an RPC record: a handler that throws is reported and the run continues.

## What this notebook does **not** establish

- **`clear_queue` + `abort`, the UI timeout default and the less common dialog methods are not run** (`ch35-lim2`).
- **The Python client in the chapter was not executed** (`ch35-lim2`).
- **The model is scripted** — the protocol is measured, not model behaviour.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real line-framed JSONL against the shipped binary, driven by the book's `RpcClient`.
* **Evidence scope:** `shipped_binary`. The protocol is the real one.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(35))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/35-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: id matching, and `prompt` means accepted

In [3]:
RPC = pinb.driver_source("ch35-rpc.ts")
rpc = pinb.driver(RPC, label="ch35-rpc", timeout=420)
ids, acc = rpc["idMatching"], rpc["promptAccepted"]
print(pinb.md_table(
    ["check", "value"],
    [["response a command", ids["aCommand"]],
     ["response b command", ids["bCommand"]],
     ["both succeeded", ids["bothSucceeded"]],
     ["prompt disposition", acc["disposition"]],
     ["response before first run event", acc["responseBeforeRun"]]],
))

| check | value |
|---|---|
| response a command | get_state |
| response b command | get_available_models |
| both succeeded | True |
| prompt disposition | started |
| response before first run event | True |


In [4]:
pinb.show_checks([
    pinb.check("two outstanding commands are matched by id",
               ids["aCommand"] == "get_state" and ids["bCommand"] == "get_available_models", "matched by id"),
    pinb.check("a successful prompt response means accepted (disposition started)",
               acc["disposition"] == "started", "started"),
    pinb.check("the response arrives before the run starts",
               acc["responseBeforeRun"], "response first"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------------------  --------------
PASS  two outstanding commands are matched by id  matched by id
PASS  a successful prompt response means accepted (disposition started)  started
PASS  the response arrives before the run starts  response first

3/3 assertions held.


## `agent_end` is not the end

In [5]:
e = rpc["agentEnd"]
print(pinb.md_table(
    ["check", "value"],
    [["agent_end before agent_settled", e["agentEndBeforeSettled"]],
     ["last record", e["last"]]],
))

| check | value |
|---|---|
| agent_end before agent_settled | True |
| last record | agent_settled |


In [6]:
pinb.show_checks([
    pinb.check("agent_settled is what a client waits for",
               e["agentEndBeforeSettled"] and e["last"] == "agent_settled", "settled last"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------  ------------
PASS  agent_settled is what a client waits for  settled last

1/1 assertions held.


## Protocol errors: malformed JSON and unknown commands

In [7]:
m, u = rpc["malformed"], rpc["unknown"]
print(pinb.md_table(
    ["case", "success", "has id", "process still serving"],
    [["malformed JSON", m["success"], "yes" if m["hasId"] else "no", "-"],
     ["unknown command", u["unknownSuccess"], "-", "yes" if u["stillServing"] else "no"]],
))

| case | success | has id | process still serving |
|---|---|---|---|
| malformed JSON | False | no | - |
| unknown command | False | - | yes |


In [8]:
pinb.show_checks([
    pinb.check("malformed JSON gives a failed parse response with no id",
               m["success"] is False and not m["hasId"], "no id"),
    pinb.check("an unknown command is an error, not a crash",
               u["unknownSuccess"] is False and u["stillServing"], "still serving"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  -------------
PASS  malformed JSON gives a failed parse response with no id  no id
PASS  an unknown command is an error, not a crash  still serving

2/2 assertions held.


## The UI subprotocol blocks on the client

In [9]:
ui = rpc["ui"]
print(pinb.md_table(
    ["check", "value"],
    [["confirm title", ui["title"]],
     ["blocked before the answer", ui["blockedBeforeAnswer"]],
     ["notify after the answer", ui["noteMessage"]],
     ["no normal command response", ui["noNormalResponse"]]],
))

| check | value |
|---|---|
| confirm title | Clear session? |
| blocked before the answer | True |
| notify after the answer | confirmed |
| no normal command response | True |


In [10]:
pinb.show_checks([
    pinb.check("the UI request carries a title and an id",
               ui["title"] == "Clear session?", "title present"),
    pinb.check("the client blocks until it answers",
               ui["blockedBeforeAnswer"], "blocked"),
    pinb.check("a UI response does not produce a normal command response",
               ui["noNormalResponse"], "no response record"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  ------------------
PASS  the UI request carries a title and an id  title present
PASS  the client blocks until it answers  blocked
PASS  a UI response does not produce a normal command response  no response record

3/3 assertions held.


## `get_entries` as a durable cursor, and `extension_error`

In [11]:
cu, er = rpc["cursor"], rpc["extensionError"]
print(pinb.md_table(
    ["check", "value"],
    [["leaf matches the last entry id", cu["leafMatchesCursor"]],
     ["since cursor returns newer entries", cu["newerHasEntries"]],
     ["unknown cursor fails", cu["invalidCursorFails"]],
     ["extension_error event", er["event"]],
     ["error message matches", er["errorMatches"]]],
))

| check | value |
|---|---|
| leaf matches the last entry id | True |
| since cursor returns newer entries | True |
| unknown cursor fails | True |
| extension_error event | agent_start |
| error message matches | True |


In [12]:
pinb.show_checks([
    pinb.check("get_entries reports the leaf and supports a since cursor",
               cu["leafMatchesCursor"] and cu["newerHasEntries"], "cursor works"),
    pinb.check("an unknown cursor fails", cu["invalidCursorFails"], "fails"),
    pinb.check("a handler that throws is reported and the run continues",
               er["event"] == "agent_start" and er["errorMatches"], "extension_error"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  ---------------
PASS  get_entries reports the leaf and supports a since cursor  cursor works
PASS  an unknown cursor fails  fails
PASS  a handler that throws is reported and the run continues  extension_error

3/3 assertions held.


## The chapter's own tests

All fourteen RPC tests.

## The chapter's canonical evidence (shipped binary)

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [13]:
PATTERNS = 'ch35-rpc/rpc.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=420)
pinb.show_tap(tap, files='ch35-rpc/rpc.test.ts', show="response")

$ node --test ch35-rpc/rpc.test.ts
  PASS  14 passed, 0 failed, 0 skipped  in 10419 ms

  tests matching 'response':
    ok   responses repeat the command id, and two outstanding commands are matched by id, not by order
    ok   a successful prompt response means accepted, not finished: disposition is started, and the run follows
    ok   prompt while streaming without streamingBehavior is an error response; with steer it is queued
    ok   malformed JSON produces a parse response with no request id
    ok   an unknown command is an error response, not a crash
    ok   extension UI subprotocol: a dialog blocks on the client, and the response uses the request id


## Your turn: predict, then run

**Predict first.** A `prompt` response says `disposition: "started"`. What does a
second prompt sent while the first is running return if it says `steer`?

**Then change one input.** Send `get_entries` with `since` set to the **first**
entry id instead of the last. How many entries come back?

**Predict the boundary.** The UI subprotocol blocks the client. What happens if
the client never answers — is there a timeout, and what value does the dialog
resolve to?

In [14]:
print("Predictions:")
print("  1. A second prompt with steer returns success with disposition queued.")
print("  2. since = first entry id returns every entry after it (all but the first).")
print("  3. No answer: the client blocks until a timeout resolves the dialog (documented, not run).")
print()
print("Observed above:")
print(f"  id matching: a={ids['aCommand']}, b={ids['bCommand']}, both ok={ids['bothSucceeded']}")
print(f"  prompt disposition: {acc['disposition']}, response first={acc['responseBeforeRun']}")
print(f"  agent_end before settled={e['agentEndBeforeSettled']}, last={e['last']}")
print(f"  ui: blocked={ui['blockedBeforeAnswer']}, note={ui['noteMessage']}, no normal response={ui['noNormalResponse']}")
print()
assert ids["bothSucceeded"] and acc["disposition"] == "started" and acc["responseBeforeRun"]
assert e["agentEndBeforeSettled"] and e["last"] == "agent_settled"
assert m["success"] is False and not m["hasId"]
assert ui["blockedBeforeAnswer"] and ui["noNormalResponse"]
assert cu["leafMatchesCursor"] and cu["invalidCursorFails"]
print("held: prompt means accepted; agent_settled is the end; the UI subprotocol blocks and matches by id")

Predictions:
  1. A second prompt with steer returns success with disposition queued.
  2. since = first entry id returns every entry after it (all but the first).
  3. No answer: the client blocks until a timeout resolves the dialog (documented, not run).

Observed above:
  id matching: a=get_state, b=get_available_models, both ok=True
  prompt disposition: started, response first=True
  agent_end before settled=True, last=agent_settled
  ui: blocked=True, note=confirmed, no normal response=True

held: prompt means accepted; agent_settled is the end; the UI subprotocol blocks and matches by id


## Interpretation

RPC has three contracts, and the chapter shows each:

| Contract | What it means | The boundary |
|---|---|---|
| **Responses** | Repeat the command id | Matched by id, not order |
| **`prompt`** | Accepted, not finished | `disposition: "started"`; wait for `agent_settled` |
| **Session events** | Carry no command id | Only responses have ids |
| **UI subprotocol** | Request/answer pair | Blocks the client; matched by id |
| **Errors** | Parse failure (no id), unknown command (error), handler throw (`extension_error`) | The process keeps serving |

Practical rules:
1. **Wait for `agent_settled`** — a `prompt` response is acceptance, not completion.
2. **Match responses by id** — do not assume order.
3. **Participate in the UI subprotocol** — answer request ids, or the client blocks.
4. **Use `since` as a durable cursor** — entry ids survive client restarts.
5. **Distinguish the error kinds** — a parse error has no id; a handler throw is an `extension_error` record.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real line-framed JSONL against the shipped binary, driven by the book's `RpcClient`.

**Evidence scope:** `shipped_binary`. The protocol is the real one.

### What was read or run

- **Ran** `drivers/ch35-rpc.ts`: eight RPC scenarios (id matching, prompt acceptance, agent_settled, malformed JSON, unknown command, UI subprotocol, entry cursor, extension_error).
- **Ran** `ch35-rpc/rpc.test.ts` (14 tests).
- **Read** `examples/evidence.json`, chapter 35 rows (14 claims).

### Limitations

- **`clear_queue` + `abort`, the UI timeout default and the less common dialog methods are not run** (`ch35-lim2`).
- **The Python client in the chapter was not executed** (`ch35-lim2`).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
